# Scotland wastewater admissions prototype

Clean the Scotland national wastewater RNA series, align it with the repo's UKHSA GP/admission-style outcome files, and run leakage-safe expanding-origin forecasts using positive wastewater lags only. The geography of the automatically inferred UKHSA outcome should be checked before treating the result as a Scotland-specific model.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from wastewater.io import find_repo_root
from wastewater.ukhsa import build_ukhsa_series_catalogue, chart_to_series

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## Configuration

The default source is `data/raw/scotland_wastewater_covid19_national.csv`. Positive lags mean week `t` admissions are predicted from wastewater in earlier weeks only.

In [ ]:
COUNTRY = 'Scotland'
PATHOGEN = 'SARS-CoV-2'
WASTEWATER_FILE = 'data/raw/scotland_wastewater_covid19_national.csv'
LAGS = [1, 2, 3, 4]
MIN_TRAIN_SIZE = 20
RIDGE_ALPHA = 1.0
SPIKE_QUANTILE = 0.90
OUTCOME_FILES = None  # set manually after inspecting the UKHSA catalogue if needed

## Clean the wastewater data

The Scotland file has `SevenDayEnding` dates encoded as `YYYYMMDD` and a `WastewaterRNA` value column.

In [ ]:
def normalise_column_names(df):
    out = df.copy()
    out.columns = (pd.Index(out.columns).astype(str).str.strip()
                   .str.replace(r'[^0-9A-Za-z]+', '_', regex=True)
                   .str.strip('_').str.lower())
    return out

def parse_yyyymmdd(series):
    text = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True).str.zfill(8)
    return pd.to_datetime(text, format='%Y%m%d', errors='coerce')

raw_path = ROOT / WASTEWATER_FILE
if not raw_path.exists():
    raise FileNotFoundError(f'Missing {raw_path.relative_to(ROOT)}. Run scripts/download_all.py first.')

raw = pd.read_csv(raw_path)
df = normalise_column_names(raw)
required = {'sevendayending', 'wastewaterrna'}
if not required.issubset(df.columns):
    raise ValueError(f'Expected columns {required}; available columns are {list(df.columns)}')

wastewater_long = pd.DataFrame({
    'date': parse_yyyymmdd(df['sevendayending']),
    'value': pd.to_numeric(df['wastewaterrna'], errors='coerce'),
    'country': COUNTRY,
    'country_code': df['country'].astype('string') if 'country' in df.columns else pd.NA,
    'pathogen': PATHOGEN,
    'source_file': WASTEWATER_FILE,
})
wastewater_long = wastewater_long.dropna(subset=['date', 'value']).sort_values('date')
wastewater_long['week'] = wastewater_long['date'].dt.to_period('W').dt.start_time

quality = pd.DataFrame([{
    'raw_rows': len(raw),
    'clean_rows': len(wastewater_long),
    'dropped_rows': len(raw) - len(wastewater_long),
    'min_date': wastewater_long['date'].min(),
    'max_date': wastewater_long['date'].max(),
    'duplicate_dates': int(wastewater_long['date'].duplicated().sum()),
    'negative_values': int((wastewater_long['value'] < 0).sum()),
    'zero_values': int((wastewater_long['value'] == 0).sum()),
}])
display(quality)
display(wastewater_long.head())

weekly_wastewater = (
    wastewater_long.groupby('week', dropna=False)
    .agg(wastewater_rna=('value', 'mean'), n_wastewater_observations=('value', 'size'))
    .reset_index().rename(columns={'week': 'period'}).sort_values('period')
)
weekly_wastewater['log1p_wastewater_rna'] = np.log1p(weekly_wastewater['wastewater_rna'].clip(lower=0))
weekly_wastewater['country'] = COUNTRY
weekly_wastewater['pathogen'] = PATHOGEN
weekly_wastewater.to_csv(PROCESSED / 'scotland_wastewater_sars_cov_2_weekly.csv', index=False)
display(weekly_wastewater.head())

## Load and align hospital-admission outcomes

The outcome is built from UKHSA chart files classified as `gp_admissions` by `wastewater.ukhsa`. For a true Scotland-only model, replace `OUTCOME_FILES` with Scotland-specific admission data when it is available.

In [ ]:
ukhsa_catalogue = build_ukhsa_series_catalogue(ROOT)
display(ukhsa_catalogue[['path', 'filename', 'series_type', 'date_column', 'value_column']].sort_values(['series_type', 'path']))

if OUTCOME_FILES is None:
    OUTCOME_FILES = ukhsa_catalogue.loc[ukhsa_catalogue['series_type'] == 'gp_admissions', 'path'].tolist()
if not OUTCOME_FILES:
    raise ValueError('No GP/admission outcome files found. Set OUTCOME_FILES manually.')

outcome_long = pd.concat([chart_to_series(ROOT, p, series_type='gp_admissions') for p in OUTCOME_FILES], ignore_index=True)
outcome_weekly = (
    outcome_long.groupby('week', dropna=False)['value'].sum(min_count=1)
    .reset_index(name='gp_admissions').rename(columns={'week': 'period'}).sort_values('period')
)
outcome_weekly['log1p_gp_admissions'] = np.log1p(outcome_weekly['gp_admissions'].clip(lower=0))

frame = pd.merge(weekly_wastewater, outcome_weekly, on='period', how='inner').sort_values('period').reset_index(drop=True)
for lag in LAGS:
    frame[f'log1p_wastewater_rna_lag{lag}'] = frame['log1p_wastewater_rna'].shift(lag)
frame.to_csv(PROCESSED / 'scotland_wastewater_gp_admissions_regression_frame.csv', index=False)
print('Aligned rows:', len(frame))
print('Complete model rows:', frame[['log1p_gp_admissions', *[f'log1p_wastewater_rna_lag{lag}' for lag in LAGS]]].dropna().shape[0])
display(frame.head())

## Leakage-safe expanding-origin ridge forecast

At each test week, the model is fit only on previous weeks. Predictor scaling is fit inside the training fold only. The rolling training mean is used as the baseline.

In [ ]:
PREDICTORS = [f'log1p_wastewater_rna_lag{lag}' for lag in LAGS]
TARGET = 'log1p_gp_admissions'

def expanding_origin_forecast(df, predictors, target, min_train_size=20, alpha=1.0, spike_quantile=0.90):
    model_df = df[['period', target, *predictors]].dropna().sort_values('period').reset_index(drop=True)
    if len(model_df) <= min_train_size:
        raise ValueError(f'Need more complete rows than min_train_size={min_train_size}; found {len(model_df)}')
    rows = []
    for i in range(min_train_size, len(model_df)):
        train = model_df.iloc[:i]
        test = model_df.iloc[[i]]
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(train[predictors], train[target])
        pred = float(model.predict(test[predictors])[0])
        train_pred = pd.Series(model.predict(train[predictors]), index=train.index)
        sigma = float((train[target] - train_pred).std(ddof=1))
        baseline = float(train[target].mean())
        actual = float(test[target].iloc[0])
        threshold = float(train[target].quantile(spike_quantile))
        score = float(100 * norm.sf((threshold - pred) / sigma)) if np.isfinite(sigma) and sigma > 0 else np.nan
        rows.append({'period': test['period'].iloc[0], 'train_end': train['period'].max(), 'n_train': len(train),
                     'actual': actual, 'prediction': pred, 'baseline_prediction': baseline,
                     'residual': actual - pred, 'abs_error': abs(actual - pred),
                     'spike_threshold': threshold, 'spike_score': score, 'actual_spike': bool(actual >= threshold)})
    return pd.DataFrame(rows)

predictions = expanding_origin_forecast(frame, PREDICTORS, TARGET, MIN_TRAIN_SIZE, RIDGE_ALPHA, SPIKE_QUANTILE)
err = predictions['actual'] - predictions['prediction']
base_err = predictions['actual'] - predictions['baseline_prediction']
rmse = float(np.sqrt(np.mean(err**2)))
baseline_rmse = float(np.sqrt(np.mean(base_err**2)))
metrics = pd.DataFrame([{
    'country': COUNTRY, 'pathogen': PATHOGEN, 'target': 'gp_admissions',
    'n_predictions': len(predictions), 'lags': ','.join(map(str, LAGS)),
    'ridge_alpha': RIDGE_ALPHA, 'rmse': rmse, 'baseline_rmse': baseline_rmse,
    'mse_skill_vs_rolling_train_mean': float(1 - rmse**2 / baseline_rmse**2) if baseline_rmse > 0 else np.nan,
    'mae': float(np.mean(np.abs(err))), 'max_abs_error': float(predictions['abs_error'].max()),
    'correlation': float(np.corrcoef(predictions['actual'], predictions['prediction'])[0, 1]) if len(predictions) > 1 else np.nan,
    'latest_spike_score': float(predictions['spike_score'].iloc[-1]),
    'latest_actual_spike': bool(predictions['actual_spike'].iloc[-1]),
}])

predictions.to_csv(PROCESSED / 'scotland_wastewater_gp_admissions_expanding_origin_predictions.csv', index=False)
metrics.to_csv(PROCESSED / 'scotland_wastewater_gp_admissions_expanding_origin_metrics.csv', index=False)
predictions.sort_values('period').tail(12).to_csv(PROCESSED / 'scotland_wastewater_hospital_spike_scores.csv', index=False)
display(metrics)
display(predictions.tail())

## Plots and interpretation checks

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(predictions['period'], predictions['actual'], label='Observed target', marker='o')
ax.plot(predictions['period'], predictions['prediction'], label='Out-of-sample prediction', marker='o')
ax.plot(predictions['period'], predictions['baseline_prediction'], label='Rolling train-mean baseline', linestyle='--')
ax.set_title('Expanding-origin forecast: admissions from lagged Scotland wastewater')
ax.set_xlabel('Week')
ax.set_ylabel('log1p(GP/admission outcome)')
ax.legend()
fig.autofmt_xdate()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(predictions['period'], predictions['spike_score'], marker='o')
ax.set_title('Wastewater-driven hospital-admission spike score')
ax.set_xlabel('Week')
ax.set_ylabel('Spike score, 0--100')
fig.autofmt_xdate()
plt.show()

display(predictions.sort_values('spike_score', ascending=False).head(20))

## Notes

- Positive `mse_skill_vs_rolling_train_mean` means the wastewater model beat the rolling training-mean baseline.
- Large `max_abs_error` values still matter even if average RMSE looks acceptable.
- This is a prototype cleaning and modelling notebook. The main next step is to swap in a Scotland-specific respiratory admission target, then repeat the same pattern for influenza and RSV wastewater sources.